## import

In [ ]:
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from time import sleep
import json
import os

## connect MongoDB

In [ ]:
client = MongoClient('mongodb://localhost:27017/')
client.drop_database('simplize')
db = client['simplize']

## get drive

In [ ]:
driver = webdriver.Chrome()
url = 'https://simplize.vn/co-phieu/nganh/tai-chinh'
driver.get(url)
sleep(2)

In [ ]:
#lấy lĩnh vực
linh_vuc = driver.find_elements(By.XPATH, "//div[contains(@class,'simplize-row css-pmt33i')]")
#list chứa lĩnh vực
collection_list = []
#loop chạy từng lĩnh vực
for chay_linh_vuc in linh_vuc:
    data = chay_linh_vuc.text
    collection_name = data
    collection = db[collection_name]
    collection.insert_one({"linhvuc": data})
    collection_list.append(data)

print(collection_list)

# Tạo dic ma_list cho từng linh_vuc
ma_lists = []  # Danh sách chứa các ma_list của từng linh_vuc
#loop từng lĩnh vực
for index in range(len(linh_vuc)):
    # Cập nhật lại danh sách linh_vuc trước khi nhấp vào phần tử
    linh_vuc = driver.find_elements(By.XPATH, "//div[contains(@class,'simplize-row css-pmt33i')]")
    linh_vuc[index].click() 
    sleep(2)
    
    # Lấy information 
    cac_ma = driver.find_elements(By.XPATH, "//div[contains(@class,'css-70qvj9')]")
    ma_list = []  # Tạo list để chứa các thông tin vừa lấy được

    for ma in cac_ma:
        ma_list.append(ma.text)  # add vào list

    print(f"Ma list for {collection_list[index]}: {ma_list}")
    ma_lists.append(ma_list)  # add ma_list vào ma_lists

    driver.back()  # back trang trước để click linh vực tiếp theo
    sleep(2)  

## cắt bớt list để giảm time chạy

In [ ]:
ma_lists

In [ ]:
to_remove1 = ('BID','CTG', 'TCB', 'VPB', 'MBB', 'ACB', 'LPB', 'HDB', 'STB', 'VIB', 'SSB', 'TPB', 'SHB', 'EIB', 'MSB', 'OCB', 'NAB', 'BAB', 'EVF', 'ABB', 'PGB', 'BVB', 'VBB', 'VAB', 'NVB', 'KLB', 'SGB','TIN')
to_remove2 = ('SSI','VND','VCI', 'VIX', 'MBS', 'FTS', 'SHS', 'BSI', 'DSE', 'CTS', 'VDS', 'ORS', 'DSC', 'AGR', 'TVS', 'BVS', 'APG', 'VFS', 'PHS', 'AAS', 'TVB', 'TCI', 'EVS', 'ABW', 'SBS', 'IVS', 'BMS', 'APS','CSI')
to_remove3 = ('PVI', 'VNR', 'BIC', 'MIG', 'BMI', 'PGI', 'PTI', 'PRE', 'ABI', 'AIC', 'BHI')

# Xóa bớt ma trong các list 
ma_lists[0] = [x for x in ma_lists[0] if x not in to_remove1]
ma_lists[1] = [y for y in ma_lists[1] if y not in to_remove2]
ma_lists[2] = [z for z in ma_lists[2] if z not in to_remove3]

In [ ]:
print(ma_lists)

In [ ]:
for ma_list in ma_lists:  # loop từng list mã 
    for ma in ma_list:  # loop từng mã c trong list hiện tại
        lich_su_gia_url = f"https://simplize.vn/co-phieu/{ma}/lich-su-gia"  # URL từng mã trong ma_list
        driver.get(lich_su_gia_url)  
        sleep(2)
        
        # get giá
        ls_gia = driver.find_elements(By.XPATH, "//tr[contains(@class,'simplize-table-row simplize-table-row-level-0')]")
        ls_gia_list = []  # Tạo list để chứa giá

        for row in ls_gia:
            row_data = {
                "date": row.find_element(By.XPATH, ".//td[1]").text,              
                "opening_price": row.find_element(By.XPATH, ".//td[2]").text,     
                "highest_price": row.find_element(By.XPATH, ".//td[3]").text,     
                "lowest_price": row.find_element(By.XPATH, ".//td[4]").text,      
                "closing_price": row.find_element(By.XPATH, ".//td[5]").text,     
                "price_change": row.find_element(By.XPATH, ".//td[6]").text,       
                "percent_change": row.find_element(By.XPATH, ".//td[7]").text,     
                "volume": row.find_element(By.XPATH, ".//td[8]").text,             
            }
            ls_gia_list.append(row_data)  # add date vào list
        
        # save file JSON
        with open(f"{ma}_lich_su_gia.json", "w", encoding='utf-8') as json_file:
            json.dump(ls_gia_list, json_file, ensure_ascii=False, indent=4) 
        
        # back về trang trước để tiếp tục vào mã tiếp theo
        driver.back()

In [ ]:
# loop qua từng phần tử của ma_lists
for index, ma_list in enumerate(ma_lists):
    collection_name = collection_list[index]  # get tên lĩnh vực tương ứng
    collection = db[collection_name]  # Truy cập vào collection tương ứng

    # loop qua từng mã  trong ma_list
    for ma in ma_list:
        json_file_path = f"{ma}_lich_su_gia.json"

        # check file có tồn tại
        if os.path.exists(json_file_path):
            with open(json_file_path, "r", encoding='utf-8') as json_file:
                ls_gia_list = json.load(json_file)  

                # Cập nhật object của mã 
                collection.update_one(
                    {"linhvuc": collection_name},  # Tìm đối tượng có giá trị là collection_name
                    {"$set": {f"lich_su_gia.{ma}": ls_gia_list}},  # Thêm lịch sử giá vào theo mã 
                    upsert=True  # Tạo mới nếu ko có
                )

In [ ]:
driver.quit()